# R1-A1-S1 - Redes Neuronales Básicas desde cero

**Estudiante:** Carlos Mario Salvatore Ocampo Aguilar  
**Asignatura:** 608142 - Deep Learning: Conceptos  
**Fecha:** 11 de septiembre de 2026

## Objetivo

Implementar y comprobar, usando exclusivamente Python y NumPy, tres modelos neuronales fundamentales:

1. un perceptrón para clasificación lineal;
2. una red neuronal de una capa con activación sigmoide;
3. una red multicapa *vainilla* capaz de resolver XOR.

Cada implementación muestra la propagación hacia adelante, el aprendizaje de pesos y la generación de predicciones. No se utilizan TensorFlow, Keras, PyTorch ni otras librerías especializadas de Deep Learning.

## Guía de ejecución

1. Abrir el archivo en Google Colab o Jupyter Notebook.
2. Ejecutar las celdas en orden, desde la preparación del entorno hasta las pruebas automáticas.
3. Leer primero la explicación en Markdown y después ejecutar el código asociado.
4. Revisar las predicciones, la pérdida y la exactitud impresas al final de cada actividad.
5. Confirmar que la celda de pruebas automáticas termine con el mensaje de aprobación.

> **Importante:** se debe usar **Entorno de ejecución > Ejecutar todas** para conservar el orden de las variables.

## 1. Preparación del entorno

### Paso 1. Cargar las librerías

En este laboratorio solo se necesitan dos módulos:

| Librería | Instrucción | Uso en el notebook |
|---|---|---|
| `sys` | `import sys` | Consultar la versión de Python con la que se ejecuta el archivo. |
| `NumPy` | `import numpy as np` | Crear arreglos, realizar productos matriciales, inicializar pesos y calcular gradientes. |

El alias `np` permite escribir instrucciones breves como `np.array`, `np.mean` y `np.random.default_rng`. No se requiere instalar NumPy en Google Colab porque ya viene disponible.

### Paso 2. Configurar la presentación numérica

`np.set_printoptions(precision=4, suppress=True)` muestra cuatro decimales y evita notación científica innecesaria. Esta instrucción solo cambia la visualización; no modifica los cálculos.

### Paso 3. Definir la semilla

`SEED = 42` fija el punto inicial del generador aleatorio. Con la misma semilla se obtienen los mismos pesos iniciales y el experimento puede repetirse.

### Paso 4. Crear la función de exactitud

`accuracy` compara la clase real con la predicha y calcula la proporción de coincidencias. Un resultado de `1.0` equivale al 100 % de aciertos.

### Paso 5. Verificar el entorno

La última parte de la celda imprime las versiones de Python y NumPy y confirma la semilla empleada.

In [ ]:
# 1) sys permite consultar la versión del intérprete de Python.
import sys
# 2) NumPy se importa con el alias np para trabajar con vectores y matrices.
import numpy as np

# 3) Configuración visual: cuatro decimales y sin notación científica.
np.set_printoptions(precision=4, suppress=True)
# 4) Semilla reproducible para inicializar siempre los mismos pesos aleatorios.
SEED = 42

# 5) Función auxiliar para medir la proporción de predicciones correctas.
def accuracy(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred = np.asarray(y_pred).reshape(-1).astype(int)
    return float(np.mean(y_true == y_pred))

# 6) Comprobación del entorno antes de iniciar las actividades.
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Semilla:", SEED)

Python: 3.12.14
NumPy: 2.3.5
Semilla: 42


## 2. Perceptrón

El perceptrón calcula $z=\mathbf{x}\cdot\mathbf{w}+b$ y aplica una función escalón. Durante el entrenamiento, si la clase estimada no coincide con la real, actualiza pesos y sesgo mediante:

$$\mathbf{w}\leftarrow\mathbf{w}+\eta(y-\hat y)\mathbf{x},\qquad b\leftarrow b+\eta(y-\hat y).$$

AND y OR son problemas linealmente separables; por eso un perceptrón puede aprenderlos.

### Desarrollo paso a paso

1. **Inicializar el modelo:** comenzar con pesos y sesgo iguales a cero.
2. **Calcular la suma ponderada:** multiplicar cada entrada por su peso y sumar el sesgo.
3. **Aplicar el escalón:** devolver clase 1 cuando la suma sea mayor o igual a cero; en caso contrario, devolver 0.
4. **Calcular el error:** restar la predicción a la clase real.
5. **Actualizar los parámetros:** corregir pesos y sesgo usando la tasa de aprendizaje.
6. **Repetir por épocas:** detener el entrenamiento cuando no existan errores.
7. **Evaluar:** comparar las salidas aprendidas con las tablas de verdad de AND y OR.

In [ ]:
# PASO 1: definir la estructura y el comportamiento del perceptrón.
class Perceptron:
    '''Perceptrón binario entrenado con la regla clásica de actualización.'''

    def __init__(self, learning_rate=0.1, epochs=30):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = 0.0
        self.errors_per_epoch = []

    @staticmethod
    def step(z):
        return (z >= 0.0).astype(int)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        self.weights = np.zeros(X.shape[1], dtype=float)
        self.bias = 0.0

        for _ in range(self.epochs):
            errors = 0
            for xi, target in zip(X, y):
                prediction = int(self.step(np.array([xi @ self.weights + self.bias]))[0])
                update = self.learning_rate * (target - prediction)
                self.weights += update * xi
                self.bias += update
                errors += int(update != 0.0)
            self.errors_per_epoch.append(errors)
            if errors == 0:
                break
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return self.step(X @ self.weights + self.bias)

In [ ]:
# PASO 2: construir todas las combinaciones posibles de dos entradas binarias.
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_and = np.array([0, 0, 0, 1])
y_or = np.array([0, 1, 1, 1])

# PASO 3: crear y entrenar un modelo independiente para cada compuerta.
perceptron_and = Perceptron().fit(X_logic, y_and)
perceptron_or = Perceptron().fit(X_logic, y_or)

# PASO 4: generar las predicciones con los pesos ya aprendidos.
pred_and = perceptron_and.predict(X_logic)
pred_or = perceptron_or.predict(X_logic)

# PASO 5: mostrar resultados, parámetros finales y exactitud.
print("Entradas:\n", X_logic.astype(int))
print("AND esperado:", y_and, "predicho:", pred_and)
print("OR  esperado:", y_or, "predicho:", pred_or)
print("Pesos AND:", perceptron_and.weights, "sesgo:", round(perceptron_and.bias, 4))
print("Pesos OR :", perceptron_or.weights, "sesgo:", round(perceptron_or.bias, 4))
print("Exactitud AND:", accuracy(y_and, pred_and))
print("Exactitud OR :", accuracy(y_or, pred_or))

Entradas:
 [[0 0]
 [0 1]
 [1 0]
 [1 1]]
AND esperado: [0 0 0 1] predicho: [0 0 0 1]
OR  esperado: [0 1 1 1] predicho: [0 1 1 1]
Pesos AND: [0.2 0.1] sesgo: -0.2
Pesos OR : [0.1 0.1] sesgo: -0.1
Exactitud AND: 1.0
Exactitud OR : 1.0


### Interpretación

Los pesos aprendidos definen una frontera lineal. En AND, solo la combinación $(1,1)$ queda en el lado positivo; en OR, basta con que una entrada sea uno. La convergencia y el 100 % de exactitud confirman que la regla de actualización funciona para estos conjuntos.

## 3. Red neuronal de una capa

Esta red contiene una neurona sigmoide. A diferencia del perceptrón, genera una probabilidad continua:

$$\hat y=\sigma(XW+b),\qquad \sigma(z)=\frac{1}{1+e^{-z}}.$$

Los parámetros se optimizan con descenso de gradiente y entropía cruzada binaria. Toda la minibase se procesa al mismo tiempo mediante multiplicación matricial.

### Desarrollo paso a paso

1. Definir la función sigmoide para convertir la suma ponderada en una probabilidad entre 0 y 1.
2. Definir la entropía cruzada binaria para medir la diferencia entre probabilidades y clases reales.
3. Inicializar los pesos con valores aleatorios reproducibles y el sesgo en cero.
4. Ejecutar la propagación hacia adelante: `X @ W + b` y luego la sigmoide.
5. Calcular los gradientes de pesos y sesgo.
6. Restar los gradientes multiplicados por la tasa de aprendizaje.
7. Repetir el proceso y almacenar la pérdida de cada época.
8. Convertir probabilidades en clases mediante el umbral 0,5 y medir la exactitud.

In [ ]:
# PASO 1: activación sigmoide para producir probabilidades.
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

# PASO 2: función de pérdida para clasificación binaria.
def binary_cross_entropy(y, probabilities):
    eps = 1e-12
    p = np.clip(probabilities, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))


# PASO 3: modelo con una sola neurona entrenable.
class SingleLayerNetwork:
    '''Clasificador binario: dos entradas y una neurona sigmoide.'''

    def __init__(self, input_dim, learning_rate=0.5, epochs=3000, seed=SEED):
        rng = np.random.default_rng(seed)
        self.weights = rng.normal(0.0, 0.2, size=(input_dim, 1))
        self.bias = np.zeros((1, 1))
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.loss_history = []

    def forward(self, X):
        return sigmoid(X @ self.weights + self.bias)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        m = X.shape[0]

        for _ in range(self.epochs):
            probabilities = self.forward(X)
            dz = probabilities - y
            dw = X.T @ dz / m
            db = np.sum(dz, axis=0, keepdims=True) / m
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            self.loss_history.append(binary_cross_entropy(y, probabilities))
        return self

    def predict_proba(self, X):
        return self.forward(np.asarray(X, dtype=float))

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int).reshape(-1)

In [ ]:
# PASO 4: crear y entrenar la neurona con la compuerta OR.
single_layer = SingleLayerNetwork(input_dim=2).fit(X_logic, y_or)
# PASO 5: obtener probabilidades y convertirlas en clases con umbral 0,5.
or_probabilities = single_layer.predict_proba(X_logic).reshape(-1)
or_predictions = single_layer.predict(X_logic)

# PASO 6: revisar resultados, reducción de la pérdida y exactitud.
print("Probabilidades OR:", np.round(or_probabilities, 4))
print("Predicciones OR   :", or_predictions)
print("Pérdida inicial   :", round(single_layer.loss_history[0], 6))
print("Pérdida final     :", round(single_layer.loss_history[-1], 6))
print("Exactitud         :", accuracy(y_or, or_predictions))
print("Forma de W        :", single_layer.weights.shape)

Probabilidades OR: [0.0136 0.9946 0.9946 1.    ]
Predicciones OR   : [0 1 1 1]
Pérdida inicial   : 0.732051
Pérdida final     : 0.00616
Exactitud         : 1.0
Forma de W        : (2, 1)


### Interpretación

La pérdida disminuye durante el entrenamiento y las probabilidades quedan a lados opuestos del umbral 0,5. La forma `(2, 1)` de la matriz de pesos confirma que las cuatro observaciones se procesan de manera vectorizada y que cada una de las dos características contribuye a una única salida.

## 4. Red neuronal multicapa *vainilla*

XOR no es linealmente separable. Se implementa una red $2\rightarrow4\rightarrow1$: cuatro neuronas ocultas con activación hiperbólica y una salida sigmoide. La capa oculta crea una representación no lineal; después, la salida combina esa representación para clasificar XOR.

La retropropagación se programa explícitamente con NumPy. Para sigmoide con entropía cruzada, el gradiente de salida se simplifica a $(\hat y-y)/m$.

### Desarrollo paso a paso

1. Crear matrices `W1` y `W2` con una semilla y sesgos `b1` y `b2` iguales a cero.
2. Multiplicar las entradas por `W1` y aplicar `tanh` en las cuatro neuronas ocultas.
3. Multiplicar las activaciones ocultas por `W2` y aplicar sigmoide en la salida.
4. Calcular la pérdida de la predicción.
5. Propagar el error desde la salida hacia la capa oculta mediante la regla de la cadena.
6. Actualizar las cuatro matrices o vectores de parámetros.
7. Repetir durante 10 000 épocas.
8. Evaluar las cuatro combinaciones de XOR y comprobar que la pérdida final sea menor que la inicial.

In [ ]:
# PASO 1: definir una red multicapa con arquitectura 2-4-1.
class VanillaMLP:
    '''Red 2-4-1 con tanh, sigmoide y retropropagación manual.'''

    def __init__(self, input_dim=2, hidden_dim=4, learning_rate=0.8,
                 epochs=10000, seed=7):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0.0, np.sqrt(1 / input_dim), (input_dim, hidden_dim))
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = rng.normal(0.0, np.sqrt(1 / hidden_dim), (hidden_dim, 1))
        self.b2 = np.zeros((1, 1))
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.loss_history = []

    def forward(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = np.tanh(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = sigmoid(z2)
        return a1, a2

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        m = X.shape[0]

        for _ in range(self.epochs):
            a1, a2 = self.forward(X)
            self.loss_history.append(binary_cross_entropy(y, a2))

            dz2 = (a2 - y) / m
            dW2 = a1.T @ dz2
            db2 = np.sum(dz2, axis=0, keepdims=True)
            da1 = dz2 @ self.W2.T
            dz1 = da1 * (1.0 - a1**2)
            dW1 = X.T @ dz1
            db1 = np.sum(dz1, axis=0, keepdims=True)

            self.W2 -= self.learning_rate * dW2
            self.b2 -= self.learning_rate * db2
            self.W1 -= self.learning_rate * dW1
            self.b1 -= self.learning_rate * db1
        return self

    def predict_proba(self, X):
        return self.forward(np.asarray(X, dtype=float))[1]

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int).reshape(-1)

In [ ]:
# PASO 2: definir la tabla de verdad de XOR y entrenar la red.
y_xor = np.array([0, 1, 1, 0])
mlp = VanillaMLP().fit(X_logic, y_xor)
# PASO 3: calcular probabilidades y clases para las cuatro entradas.
xor_probabilities = mlp.predict_proba(X_logic).reshape(-1)
xor_predictions = mlp.predict(X_logic)

# PASO 4: interpretar la pérdida, exactitud y formas de las matrices.
print("XOR esperado      :", y_xor)
print("Probabilidades XOR:", np.round(xor_probabilities, 4))
print("Predicciones XOR  :", xor_predictions)
print("Pérdida inicial   :", round(mlp.loss_history[0], 6))
print("Pérdida final     :", round(mlp.loss_history[-1], 6))
print("Exactitud         :", accuracy(y_xor, xor_predictions))
print("Formas W1 y W2    :", mlp.W1.shape, mlp.W2.shape)

XOR esperado      : [0 1 1 0]
Probabilidades XOR: [0.     0.9997 0.9996 0.0004]
Predicciones XOR  : [0 1 1 0]
Pérdida inicial   : 0.706405
Pérdida final     : 0.00025
Exactitud         : 1.0
Formas W1 y W2    : (2, 4) (4, 1)


### Interpretación

La red multicapa separa correctamente los cuatro casos de XOR, algo imposible para una sola frontera lineal. `W1` tiene forma `(2, 4)` y transforma dos características en cuatro activaciones ocultas; `W2` tiene forma `(4, 1)` y las combina en una salida. La reducción de la pérdida evidencia que la retropropagación ajustó las dos capas.

## 5. Pruebas automáticas

Estas aserciones verifican que los tres modelos sean funcionales y ejecutables. Si una predicción cambia de forma incorrecta, el notebook se detiene con un error en lugar de ocultar el fallo.

La validación se realiza en tres pasos: (1) exigir 100 % de exactitud para AND y OR con el perceptrón, (2) exigir 100 % para la red de una capa y XOR, y (3) confirmar que el aprendizaje de la red multicapa redujo la pérdida. Si no aparece ninguna excepción, todas las condiciones se cumplieron.

In [ ]:
assert accuracy(y_and, pred_and) == 1.0
assert accuracy(y_or, pred_or) == 1.0
assert accuracy(y_or, or_predictions) == 1.0
assert accuracy(y_xor, xor_predictions) == 1.0
assert mlp.loss_history[-1] < mlp.loss_history[0]

print("Todas las pruebas pasaron correctamente.")

Todas las pruebas pasaron correctamente.


## 6. Conclusiones

- El perceptrón aprende fronteras lineales mediante una regla local basada en el error.
- Una neurona sigmoide añade una interpretación probabilística y puede entrenarse de forma vectorizada con descenso de gradiente.
- XOR demuestra la limitación de una sola capa lineal; la capa oculta con `tanh` introduce la no linealidad necesaria.
- NumPy es suficiente para programar propagación hacia adelante, cálculo de pérdida y retropropagación, lo que permite comprender la lógica interna antes de usar librerías de alto nivel.
- Las semillas, las formas de las matrices y las aserciones hacen que el trabajo sea reproducible y verificable.
